In [1]:
import pandas as pd

df = pd.read_csv("../../data/train_labels.csv")
print(df.shape)
print(df.columns.tolist())
print("Farklı tomogram sayısı:", df["tomo_id"].nunique())
df.head()

(737, 10)
['row_id', 'tomo_id', 'Motor axis 0', 'Motor axis 1', 'Motor axis 2', 'Array shape (axis 0)', 'Array shape (axis 1)', 'Array shape (axis 2)', 'Voxel spacing', 'Number of motors']
Farklı tomogram sayısı: 648


,row_id,tomo_id,Motor axis 0,Motor axis 1,Motor axis 2,Array shape (axis 0),Array shape (axis 1),Array shape (axis 2),Voxel spacing,Number of motors
0,0,tomo_003acc,-1.0,-1.0,-1.0,500,1912,1847,6.5,0
1,1,tomo_00e047,169.0,546.0,603.0,300,959,928,15.6,1
2,2,tomo_00e463,235.0,403.0,137.0,500,924,956,19.7,6
3,3,tomo_00e463,243.0,363.0,153.0,500,924,956,19.7,6
4,4,tomo_00e463,222.0,379.0,144.0,500,924,956,19.7,6


In [2]:
import sys
sys.path.append("../..")

from src import data

labels = data.load_labels("../../data/train_labels.csv")   # her satır bir motor
tomos = data.tomogram_table(labels)                        # her satır bir tomogram

print(len(labels), len(tomos))
tomos.head()

737 648


,tomo_id,shape_z,shape_y,shape_x,voxel_spacing,n_motors,has_motor
0,tomo_003acc,500,1912,1847,6.5,0,False
1,tomo_00e047,300,959,928,15.6,1,True
2,tomo_00e463,500,924,956,19.7,6,True
3,tomo_01a877,300,960,928,13.1,1,True
4,tomo_02862f,300,959,928,15.6,1,True


In [3]:
tomos["has_motor"].value_counts()

has_motor
True     362
False    286
Name: count, dtype: int64

**Observation:** 362 of the 648 training tomograms (56%) contain at least one motor and 286 (44%) contain none, so the two classes are fairly balanced.

In [4]:
tomos["n_motors"].value_counts().sort_index()

n_motors
0     286
1     313
2      30
3       6
4       9
6       3
10      1
Name: count, dtype: int64

In [5]:
tomos["n_motors"].sum()

np.int64(455)

**Observation:** Most tomograms with a motor contain exactly one: 313 of 362 (86%). The other 49 contain between 2 and 10 motors. Since a submission has only one predicted point per tomogram, a model only needs to find one of the motors in these tomograms (this is how our F2 function in `src/utils.py` scores them). The `n_motors` column sums to 455, but the CSV has coordinates for only 451 motors, so 4 motors are counted but not labelled.